In [1]:
from openai import OpenAI

# Step 1: Initialize the local Ollama client
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# Step 2: Define your prompts
system_prompt = (
    "Summarise the contents of the following email. "
    "Format the output using clear Markdown: use a bold title, "
    "line breaks, and bullet points for the key takeaways."
)

user_prompt = """
Subject: Project Timeline Adjustments

Hi Team,
I wanted to give everyone a heads-up that our launch date has been moved to next Friday. 
The development team needs an extra two days to clear out some critical bugs discovered during QA. 
Please sync with marketing to make sure the ad campaigns align with this new date. 
Additionally, we will hold a brief check-in tomorrow at 10 AM to answer any questions.
Best,
Mark
"""

# Step 3: Make the messages list
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Step 4: Call the local model
response = ollama.chat.completions.create(
    model="llama3.2",
    messages=messages
)

# Step 5: Print the result
print(response.choices[0].message.content)

**Project Timeline Adjustments**

* The launch date for the project has been moved to **next Friday**
* An additional two days have been needed to address some critical bugs discovered during QA
* A meeting with marketing is required to ensure that ad campaigns are updated and aligned with the new launch date

For further discussion:

* Brief check-in scheduled for tomorrow at 10 AM


In [ ]:
# imports
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

# Constants
MODEL = "llama3.2"

# A class to represent a Webpage
class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        
        # Remove irrelevant elements
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""

# Define our system prompt
system_prompt = (
    "You are an assistant that analyzes the contents of a website "
    "and provides a short summary, ignoring text that might be navigation related. "
    "Respond in markdown."
)

# A function that writes a User Prompt
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}. "
    user_prompt += "The contents of this website is as follows; "
    user_prompt += "please provide a short summary of this website in markdown. "
    user_prompt += "If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

# Create the messages list for the model
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

# Call the Ollama function
def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

# A function to display this nicely in Jupyter output
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

# Finally: Call the function for your website
display_summary("https://wiseinvesting.com")